pip install opencv-python

# Compressão de todas as imagens em determinada pasta

In [10]:
import os
import time
import winsound
from PIL import Image

try:
    from pillow_heif import register_heif_opener
    register_heif_opener()
    HEIF_SUPPORT = True
except ImportError:
    HEIF_SUPPORT = False
    print("[AVISO] pillow-heif não instalado. Arquivos .heic/.heif serão ignorados.")

def format_size(size_bytes):
    if size_bytes >= 1024 * 1024:
        return f"{size_bytes / (1024*1024):.2f} MB"
    elif size_bytes >= 1024:
        return f"{size_bytes / 1024:.2f} KB"
    else:
        return f"{size_bytes} B"

def compress_images(directory=None, quality=70, png_compress_level=9, max_width=None, max_height=None):
    start_time = time.perf_counter()
    erros = []
    processados = 0
    substituidos = 0
    mantidos = 0
    total_original = 0
    total_comprimido = 0

    try:
        if directory:
            print(f"[INFO] Acessando diretório: {directory}")
            os.chdir(directory)
        else:
            print("[INFO] Usando diretório atual")

        files = os.listdir()
        ext_supported = ('.jpg', '.jpeg', '.png')
        if HEIF_SUPPORT:
            ext_supported += ('.heic', '.heif')
        images = [f for f in files if f.lower().endswith(ext_supported)]
        print(f"[INFO] Encontradas {len(images)} imagens")
        print(f"[CONFIG] Qualidade: {quality} | PNG compress: {png_compress_level}")
        if max_width or max_height:
            print(f"[CONFIG] Redimensionar para max {max_width or 'inf'}x{max_height or 'inf'} px")

        if not images:
            print("[AVISO] Nenhuma imagem suportada.")
            return

        for image in images:
            original_size = os.path.getsize(image)
            total_original += original_size
            base_name, ext = os.path.splitext(image)
            ext = ext.lower()
            temp_name = f"_temp_{base_name}{ext}"

            try:
                with Image.open(image) as img:
                    # Redimensionar se solicitado
                    if max_width or max_height:
                        w, h = img.size
                        if (max_width and w > max_width) or (max_height and h > max_height):
                            ratio = min(max_width/w if max_width else 1, max_height/h if max_height else 1)
                            new_size = (int(w*ratio), int(h*ratio))
                            img = img.resize(new_size, Image.Resampling.LANCZOS)

                    # Salvar conforme formato
                    if ext in ('.jpg', '.jpeg'):
                        img.save(temp_name, format='JPEG', quality=quality,
                                 optimize=True, progressive=True, subsampling=0)
                        final_ext = ext
                    elif ext == '.png':
                        img.save(temp_name, format='PNG', optimize=True,
                                 compress_level=png_compress_level)
                        final_ext = ext
                    elif ext in ('.heic', '.heif') and HEIF_SUPPORT:
                        # Para HEIC, salvar como JPG (melhor compressão)
                        jpg_temp = f"_temp_{base_name}.jpg"
                        img.save(jpg_temp, format='JPEG', quality=quality,
                                 optimize=True, progressive=True, subsampling=0)
                        temp_name = jpg_temp
                        final_ext = '.jpg'
                    else:
                        img.save(temp_name, optimize=True)
                        final_ext = ext

                    novo_size = os.path.getsize(temp_name)
                    if novo_size < original_size:
                        reducao = (1 - novo_size / original_size) * 100
                        if final_ext != ext:
                            # Mudou de extensão (ex: HEIC -> JPG)
                            os.remove(image)
                            new_name = base_name + final_ext
                            os.rename(temp_name, new_name)
                            print(f"{base_name:30} | {format_size(original_size):>10} → {format_size(novo_size):>10} | {reducao:>5.1f}% redução ({ext}→{final_ext[1:]})")
                        else:
                            os.replace(temp_name, image)
                            print(f"{base_name:30} | {format_size(original_size):>10} → {format_size(novo_size):>10} | {reducao:>5.1f}% redução")
                        substituidos += 1
                        total_comprimido += novo_size
                    else:
                        os.remove(temp_name)
                        total_comprimido += original_size
                        mantidos += 1

                    processados += 1

            except Exception as e:
                erros.append(image)
                print(f"[ERRO] {image}: {str(e)}")
                total_comprimido += original_size
                continue

        tempo = time.perf_counter() - start_time
        reducao_total = (1 - total_comprimido / total_original) * 100 if total_original else 0

        print("\n" + "="*70)
        print(f"Resumo: {processados} imagens processadas em {tempo:.2f}s")
        print(f"Substituídas: {substituidos} arquivos (tamanho reduzido)")
        print(f"Mantidos (sem redução): {mantidos} arquivos")
        print(f"Tamanho original: {format_size(total_original)} → Final: {format_size(total_comprimido)}")
        print(f"Redução total: {reducao_total:.1f}%")

        if erros:
            print("\n[ARQUIVOS COM ERRO]")
            for err in erros:
                print(f"  - {err}")
        print("="*70)

    except Exception as e:
        print(f"[ERRO CRÍTICO] {str(e)}")
        raise

# ===== USO =====
directory = r"C:\Users\Rafael Bruno\Downloads\fotu"
try:
    compress_images(directory, quality=70, png_compress_level=9, max_width=1920, max_height=1080)
except Exception:
    winsound.Beep(2500, 1000)
finally:
    winsound.Beep(2500, 1000)

[INFO] Acessando diretório: C:\Users\Rafael Bruno\Downloads\fotu
[INFO] Encontradas 19 imagens
[CONFIG] Qualidade: 70 | PNG compress: 9
[CONFIG] Redimensionar para max 1920x1080 px
20221007_173318                |    1.83 MB →  239.32 KB |  87.2% redução (.heic→jpg)
20221008_130141                |  655.41 KB →  178.61 KB |  72.7% redução (.heic→jpg)
20221008_130142                |    1.88 MB →  269.83 KB |  86.0% redução (.heic→jpg)
2023-01-21T14_23_45-03_00      |  387.02 KB →  235.26 KB |  39.2% redução
2023-05-14T15_23_37-03_00      |  563.00 KB →  227.83 KB |  59.5% redução
2023-05-14T15_25_02-03_00      |    1.44 MB →  368.00 KB |  75.1% redução
2023-05-14T15_28_09-03_00      |  488.36 KB →  200.96 KB |  58.9% redução
[ERRO] 20251007_202338.png: cannot identify image file '20251007_202338.png'
20251216_214220                |   20.14 MB →   90.53 KB |  99.6% redução
20251216_214439                |   11.48 MB →  186.64 KB |  98.4% redução
IMG-20250817-WA0030            |  460.91